# LangChain 한국어 입문 실습 노트북

## 범위

- OpenRouter 기반 LangChain 채팅 모델 호출
- 기본 체인 구성: `ChatPromptTemplate | 모델 | StrOutputParser()`
- 반복 처리: `batch()`
- 구조화 후처리: JSON 파싱
- 단계 연결: 이전 출력 -> 다음 입력
- 로컬 GPU 임베딩 기반 RAG

## 모델 설정

- 기본 라우터: `openai/gpt-oss-20b:free`
- 기본 환경 변수: `OPENROUTER_MODEL`
- RAG 임베딩: `HuggingFaceEmbeddings`
- RAG 디바이스: `cuda`


## 참고 자료

- LangChain `ChatOpenAI`: https://docs.langchain.com/oss/python/integrations/chat/openai
- LangChain Structured Output: https://python.langchain.com/docs/how_to/structured_output/
- LangChain Sentence Transformers: https://docs.langchain.com/oss/python/integrations/embeddings/sentence_transformers
- OpenRouter Quickstart: https://openrouter.ai/docs/quickstart
- OpenRouter API 개요: https://openrouter.ai/docs/api/reference/overview


## 0. 설치와 실행 준비

- 설치:

```bash
uv sync
```

- 필수 환경 변수: `OPENROUTER_API_KEY`
- 선택 환경 변수: `OPENROUTER_BASE_URL`
- 선택 환경 변수: `OPENROUTER_MODEL`
- 설정 파일 로드 순서: `.env-example`
- 설정 파일 로드 순서: `.env`
- 설정 파일 로드 순서: OS 환경 변수
- 기본값: `OPENROUTER_BASE_URL=https://openrouter.ai/api/v1`
- 기본값: `OPENROUTER_MODEL=openai/gpt-oss-20b:free`
- RAG 전제: `langchain-huggingface`
- RAG 전제: `sentence-transformers`
- RAG 전제: CUDA 사용 가능 GPU
- RAG 전제: CUDA 지원 PyTorch
- 주의: 무료 엔드포인트는 입력이 로깅될 수 있음
- 주의: 민감한 데이터 입력 금지


In [11]:
import json
import re
import sys
import textwrap
from pathlib import Path

import torch
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

project_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "langchain_cookbook" / "openrouter_setup.py").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError("src/langchain_cookbook/openrouter_setup.py 파일을 찾지 못했습니다.")

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from langchain_cookbook.openrouter_setup import load_openrouter_settings, make_chat_model

settings = load_openrouter_settings()
OPENROUTER_BASE_URL = settings.base_url
CHAT_MODEL = settings.model


In [12]:
llm = make_chat_model()

print(f"chat 모델: {CHAT_MODEL}")
print(f"base URL: {OPENROUTER_BASE_URL}")


chat 모델: qwen/qwen3.6-plus:free
base URL: https://openrouter.ai/api/v1


## 1. 단일 모델 호출

- 목표:
  - `ChatOpenAI` 객체 생성
  - `invoke()` 호출 결과 확인
- 확인 항목:
  - 응답 객체 생성 여부
  - `content` 출력 형식


In [3]:
first_response = llm.invoke(
    "LangChain이 무엇인지 한국어로 3문장만 사용해서 아주 쉽게 설명해줘."
)

print(first_response.content)


LangChain은 인공지능 언어 모델을 쉽게 연결하고 활용할 수 있도록 도와주는 개발 도구입니다. 이 도구를 사용하면 복잡한 코딩 없이도 챗봇이나 문서 요약 같은 AI 프로그램을 빠르게 만들 수 있습니다. 마치 레고 블록처럼 필요한 기능을 조립하여 나만의 인공지능 서비스를 간편하게 완성할 수 있게 해줍니다.


## 2. 기본 체인 패턴

```python
prompt | model | parser
```

- `prompt`:
  - 입력 구조 정의
  - 시스템 지시와 사용자 입력 분리
- `model`:
  - LLM 호출 실행
- `parser`:
  - 응답을 문자열 또는 구조화 데이터로 변환
- 현재 예제 구성:
  - `ChatPromptTemplate`
  - `llm`
  - `StrOutputParser()`


In [4]:
study_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 랭체인 입문 튜터입니다. 설명은 쉽고 짧게 하고, 마지막 줄에 한 줄 실습 팁을 추가하세요.",
        ),
        (
            "human",
            "주제: {topic}\n학습자 수준: {level}\n위 조건에 맞는 설명을 작성해줘.",
        ),
    ]
)

study_chain = study_prompt | llm | StrOutputParser()

study_result = study_chain.invoke(
    {"topic": "PromptTemplate와 Output Parser", "level": "LangChain을 처음 보는 사람"}
)

print(study_result)


**PromptTemplate**은 LLM에게 보낼 프롬프트를 미리 설계하는 ‘양식’입니다. `{변수}`를 넣어두면 코드에서 값을 주입해 매번 다른 상황에 맞는 프롬프트를 자동으로 생성할 수 있어 재사용성이 높습니다.

**Output Parser**는 LLM이 뱉은 자유형 텍스트를 파이썬에서 바로 쓸 수 있는 구조화된 데이터(JSON, 딕셔너리, 리스트 등)로 변환해 주는 도구입니다. 형식이 맞지 않으면 자동으로 재요청하거나 오류를 잡아주는 기능도 기본으로 지원합니다.

둘을 함께 쓰면 “입력은 동적으로, 출력은 구조적으로” LLM을 실제 앱에 안정적으로 붙일 수 있습니다.

💡 **실습 팁:** `PromptTemplate.from_template("설명해줘: {topic}")`로 템플릿을 만들고, `| JsonOutputParser()`를 파이프(`|`)로 연결해 바로 실행해 보세요.


## 3. 반복 처리

- 사용 API: `batch()`
- 목적:
  - 동일 체인을 여러 입력에 반복 적용
- 적합한 작업:
  - 분류
  - 요약
  - 초안 생성
- 현재 예제:
  - 질문 목록 분류


In [5]:
classification_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 랭체인 학습 코치입니다. 답변 형식은 반드시 '카테고리 | 이유 | 추천 다음 단계' 한 줄로만 작성하세요.",
        ),
        ("human", "질문: {question}"),
    ]
)

classification_chain = classification_prompt | llm | StrOutputParser()

questions = [
    "프롬프트 템플릿은 왜 필요한가요?",
    "여러 단계를 연결한 체인은 어떻게 설계하나요?",
    "모델 응답을 JSON처럼 다루고 싶어요.",
]

classification_results = classification_chain.batch(
    [{"question": question} for question in questions]
)

for question, result in zip(questions, classification_results):
    print(f"질문: {question}")
    print(f"결과: {result}")
    print("-" * 80)


질문: 프롬프트 템플릿은 왜 필요한가요?
결과: 프롬프트 엔지니어링 | 변수 주입을 통한 재사용성·일관성 확보 및 유지보수 용이성 제공 | LangChain의 PromptTemplate 클래스로 변수 치환 및 포맷팅 실습하기
--------------------------------------------------------------------------------
질문: 여러 단계를 연결한 체인은 어떻게 설계하나요?
결과: 체인 설계 및 워크플로우 | LangChain의 다단계 체인은 LCEL의 파이프(|) 연산자를 통해 구성 요소의 직렬 연결과 데이터 흐름을 명시적으로 제어해야 하므로 | LCEL 공식 튜토리얼을 참고해 RunnableSequence로 프롬프트-LLM-파서 파이프라인을 직접 구현해 보세요.
--------------------------------------------------------------------------------
질문: 모델 응답을 JSON처럼 다루고 싶어요.
결과: 출력 파싱 | LLM은 기본적으로 자유 형식 텍스트를 생성하므로 JSON 구조 보장을 위해 전용 파서 또는 구조화 출력 기능이 필요합니다. | JsonOutputParser 또는 with_structured_output()과 Pydantic 모델 결합 실습하기
--------------------------------------------------------------------------------


## 4. JSON 후처리

- 목표:
  - 모델 응답을 JSON 문자열로 제한
  - LangChain에서 파싱해 dict로 변환
- 현재 방식:
  - 프롬프트로 JSON 객체만 출력하도록 지시
  - `StrOutputParser()`로 문자열 추출
  - `RunnableLambda`로 JSON 파싱
- 현재 모델:
  - `openai/gpt-oss-20b:free`


In [6]:
def extract_json_block(text: str) -> dict:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match is None:
        raise ValueError(f"JSON 객체를 찾지 못했습니다. 원문 응답: {text}")
    return json.loads(match.group())


analysis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 LangChain 학습 질문 분석기입니다. 반드시 JSON 객체 하나만 반환하세요. 추가 설명은 금지합니다.",
        ),
        (
            "human",
            "질문: {question}\n\n"
            "아래 스키마를 따르세요.\n"
            "{{\n"
            '  "difficulty": "easy | medium | hard",\n'
            '  "core_concepts": ["개념1", "개념2"],\n'
            '  "next_step": "바로 해볼 실습 1개"\n'
            "}}",
        ),
    ]
)

analysis_chain = (
    analysis_prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(extract_json_block)
)

analysis_result = analysis_chain.invoke(
    {"question": "PromptTemplate와 StrOutputParser는 각각 왜 필요하고, 언제 같이 쓰나요?"}
)

analysis_result


{'difficulty': 'medium',
 'core_concepts': ['PromptTemplate', 'StrOutputParser'],
 'next_step': 'PromptTemplate | ChatOpenAI | StrOutputParser 체인을 구성하여, 사용자 입력을 받아 LLM의 응답을 문자열로 바로 출력하는 코드를 작성하고 실행해 보세요.'}

## 5. 이전 출력 재사용

- 입력:
  - `analysis_result`
- 출력:
  - 3일 학습 계획
- 핵심 개념:
  - 체인 출력은 다음 단계 입력으로 재사용 가능
  - 임베딩 없이 다단계 워크플로우 구성 가능


In [7]:
plan_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 LangChain 입문 코치입니다. 반드시 3일 학습 계획만 작성하고, 각 날짜마다 목표와 실습 1개씩 제안하세요.",
        ),
        (
            "human",
            "난이도: {difficulty}\n핵심 개념: {core_concepts}\n추천 다음 단계: {next_step}",
        ),
    ]
)

plan_chain = plan_prompt | llm | StrOutputParser()

plan_result = plan_chain.invoke(
    {
        "difficulty": analysis_result["difficulty"],
        "core_concepts": ", ".join(analysis_result["core_concepts"]),
        "next_step": analysis_result["next_step"],
    }
)

print(plan_result)


**Day 1**
- **목표**: `PromptTemplate`의 변수 치환 구조와 동적 프롬프트 생성 원리를 이해합니다.
- **실습**: `{subject}`와 `{format}` 변수를 포함하는 `PromptTemplate`을 정의하고, `format()` 메서드로 실제 값을 주입해 최종 프롬프트 문자열을 출력하는 코드를 작성하세요.

**Day 2**
- **목표**: `StrOutputParser`가 LLM 응답 객체(`AIMessage` 등)를 순수 문자열로 변환하는 파싱 흐름을 익힙니다.
- **실습**: 임의의 LLM 응답 객체를 생성(또는 모의)한 뒤, `StrOutputParser().invoke()`를 적용해 메타데이터를 제거하고 텍스트만 추출하는 코드를 작성하세요.

**Day 3**
- **목표**: `PromptTemplate | ChatOpenAI | StrOutputParser` 체인을 구성하여, 사용자 입력을 받아 즉시 문자열 응답을 출력하는 파이프라인을 완성합니다.
- **실습**: 추천 다음 단계를 그대로 구현하세요. `input()`으로 사용자 주제를 받고, `|` 연산자로 연결된 체인에 전달한 뒤 최종 문자열 결과를 콘솔에 출력하는 스크립트를 작성하고 실행해 보세요.


## 6. 체인 결합

- 목표:
  - 분석 단계와 계획 생성 단계 연결
- 변환 함수:
  - `RunnableLambda`
- 역할:
  - 중간 dict를 다음 프롬프트 입력 형식으로 변환
- 결과:
  - 질문 1개 -> 분석 -> 계획 생성


In [8]:
def to_plan_input(result: dict) -> dict:
    return {
        "difficulty": result["difficulty"],
        "core_concepts": ", ".join(result["core_concepts"]),
        "next_step": result["next_step"],
    }


learning_plan_chain = (
    analysis_chain
    | RunnableLambda(to_plan_input)
    | plan_prompt
    | llm
    | StrOutputParser()
)

print(
    learning_plan_chain.invoke(
        {"question": "PromptTemplate, batch, JSON 파싱을 어떤 순서로 공부하면 좋을까요?"}
    )
)


# 📅 LangChain 3일 학습 계획 (Medium)

**Day 1: PromptTemplate 구조화**
- 🎯 **목표**: 변수 기반 동적 프롬프트 생성 원리와 템플릿 재사용 패턴 이해
- 💻 **실습**: `{topic}`, `{tone}`, `{length}` 변수를 포함하는 `PromptTemplate`을 작성하고, 단일 입력으로 LLM을 호출해 출력 형식과 변수 치환 동작을 확인하세요.

**Day 2: JSON Output Parsing 안정화**
- 🎯 **목표**: LLM의 자유형 응답을 스키마 기반 JSON으로 강제 변환하는 파서 적용법 습득
- 💻 **실습**: `JsonOutputParser`와 Pydantic 모델을 결합해 Day 1의 출력을 `{"title": str, "sections": list[str]}` 구조로 파싱하세요. 파싱 실패 시 기본값 반환 또는 재시도 로직을 추가해 안정성을 검증하세요.

**Day 3: Batch Processing 체인 완성**
- 🎯 **목표**: LCEL 기반 체인 구성과 `batch.invoke()`를 통한 병렬 처리 효율화 방법 익히기
- 💻 **실습**: `PromptTemplate` → `LLM` → `JsonOutputParser`를 LCEL(`|`) 연산자로 연결한 체인을 구성한 후, `chain.batch([{"topic": "...", "tone": "...", "length": "..."}, ...])`로 5개 이상의 입력을 한 번에 처리하세요. 단일 `invoke()` 대비 실행 시간, API 호출 횟수, 결과 일관성을 비교 분석하세요.

💡 **코치 팁**: 실습 환경은 `langchain-core` + `langchain-openai`(또는 호환 프로바이더) 기준으로 진행하세요. `batch.invoke()`는 내부적으로 비동기/병렬 처리를 지원하므로, 대기 시간 단축과 토큰 사용량 최적화 효과를 직접 체감하는 것이 이 단계의 핵심 목표입니다.


## 7. 로컬 GPU 임베딩으로 RAG 인덱스 만들기

- 임베딩 구현: `HuggingFaceEmbeddings`
- 임베딩 모델: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- 디바이스: `cuda`
- 벡터 저장소: `InMemoryVectorStore`
- 문서 분할: `RecursiveCharacterTextSplitter`
- 전제: CUDA 사용 가능 GPU
- 전제: CUDA 지원 PyTorch


In [9]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU를 찾지 못했습니다. CUDA 지원 PyTorch를 설치한 뒤 다시 실행하세요."
    )

HF_EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=HF_EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
)

rag_docs = [
    Document(
        page_content=textwrap.dedent(
            """
            RAG는 질문과 관련된 문서를 먼저 검색한 뒤, 검색된 문맥을 모델 입력에 함께 넣어 답변 정확도를 높이는 방식이다.
            검색 단계와 생성 단계를 분리하면 품질 저하 지점을 추적하기 쉽다.
            검색 결과의 source를 함께 보여주면 검증 가능성이 높아진다.
            """
        ).strip(),
        metadata={"source": "rag_overview.md", "topic": "RAG"},
    ),
    Document(
        page_content=textwrap.dedent(
            """
            임베딩은 텍스트를 고정 길이 벡터로 바꾸는 과정이다.
            의미가 비슷한 문장은 벡터 공간에서도 가까워지므로 유사도 검색에 사용할 수 있다.
            검색 품질은 임베딩 모델, 청크 크기, overlap 설정의 영향을 받는다.
            """
        ).strip(),
        metadata={"source": "embedding_basics.md", "topic": "Embedding"},
    ),
    Document(
        page_content=textwrap.dedent(
            """
            청크 분할은 긴 문서를 검색 가능한 단위로 나누는 과정이다.
            overlap은 문장이 경계에서 끊길 때 문맥 손실을 줄이는 데 도움이 된다.
            너무 작은 청크는 의미가 약해지고, 너무 큰 청크는 검색 정밀도가 떨어질 수 있다.
            """
        ).strip(),
        metadata={"source": "chunking.md", "topic": "Chunking"},
    ),
    Document(
        page_content=textwrap.dedent(
            """
            실무 RAG에서는 검색된 문맥 안에서만 답하도록 프롬프트를 제한해야 한다.
            문맥에 없는 내용은 모른다고 답하게 해야 환각을 줄일 수 있다.
            검색 결과와 최종 답변을 함께 로그로 남기면 운영과 디버깅이 쉬워진다.
            """
        ).strip(),
        metadata={"source": "rag_ops.md", "topic": "Operations"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=160, chunk_overlap=30)
rag_splits = splitter.split_documents(rag_docs)

vector_store = InMemoryVectorStore(hf_embeddings)
vector_store.add_documents(rag_splits)

print(f"embedding model: {HF_EMBED_MODEL}")
print("device: cuda")
print(f"원본 문서 수: {len(rag_docs)}")
print(f"분할된 청크 수: {len(rag_splits)}")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
device: cuda
원본 문서 수: 4
분할된 청크 수: 4


## 8. 검색 결과로 답변 생성

- 입력: 사용자 질문
- 검색: 상위 3개 청크
- 답변 규칙: 검색 문맥만 사용
- 답변 규칙: 문맥에 없으면 모른다고 답변
- 출력: 답변 + source


In [10]:
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        f"[source: {doc.metadata['source']}]\\n{doc.page_content}" for doc in docs
    )


retriever = vector_store.as_retriever(search_kwargs={"k": 3})

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "제공된 문맥만 사용해 답하세요. 문맥에 없으면 '모르겠습니다.'라고 답하세요. 마지막 줄에는 참고한 source를 콤마로 정리하세요.",
        ),
        (
            "human",
            "질문: {question}\\n\\n참고 문맥:\\n{context}",
        ),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

question = "RAG에서 source를 함께 보여줘야 하는 이유와, overlap이 필요한 이유를 설명해줘."
retrieved_docs = retriever.invoke(question)
context = format_docs(retrieved_docs)

print("[검색된 문서]")
for idx, doc in enumerate(retrieved_docs, start=1):
    print(f"{idx}. source={doc.metadata['source']}, topic={doc.metadata['topic']}")
    print(doc.page_content)
    print("-" * 80)

print("\n[RAG 답변]")
print(rag_chain.invoke({"question": question, "context": context}))


[검색된 문서]
1. source=rag_overview.md, topic=RAG
RAG는 질문과 관련된 문서를 먼저 검색한 뒤, 검색된 문맥을 모델 입력에 함께 넣어 답변 정확도를 높이는 방식이다.
검색 단계와 생성 단계를 분리하면 품질 저하 지점을 추적하기 쉽다.
검색 결과의 source를 함께 보여주면 검증 가능성이 높아진다.
--------------------------------------------------------------------------------
2. source=chunking.md, topic=Chunking
청크 분할은 긴 문서를 검색 가능한 단위로 나누는 과정이다.
overlap은 문장이 경계에서 끊길 때 문맥 손실을 줄이는 데 도움이 된다.
너무 작은 청크는 의미가 약해지고, 너무 큰 청크는 검색 정밀도가 떨어질 수 있다.
--------------------------------------------------------------------------------
3. source=embedding_basics.md, topic=Embedding
임베딩은 텍스트를 고정 길이 벡터로 바꾸는 과정이다.
의미가 비슷한 문장은 벡터 공간에서도 가까워지므로 유사도 검색에 사용할 수 있다.
검색 품질은 임베딩 모델, 청크 크기, overlap 설정의 영향을 받는다.
--------------------------------------------------------------------------------

[RAG 답변]
RAG에서 source를 함께 보여주는 이유는 검색 결과의 검증 가능성을 높이기 위함입니다.
Overlap이 필요한 이유는 문장이 경계에서 끊길 때 발생할 수 있는 문맥 손실을 줄여주기 위함입니다.

rag_overview.md, chunking.md


## 마무리

- 완료 항목: `ChatOpenAI` 연결
- 완료 항목: 기본 체인 구성
- 완료 항목: `batch()` 처리
- 완료 항목: JSON 후처리
- 완료 항목: 단계 연결
- 완료 항목: `HuggingFaceEmbeddings` + GPU RAG
- 전제: CUDA 사용 가능 GPU
- 전제: CUDA 지원 PyTorch
